# Commands — Submodules & Bare Repos
> *The "I've seen some things" chapter of Git. Advanced stuff.*
> *Learn this and casually drop it in standup for +5 respect points.* 🔬

---

> 💬 Junior dev: "What's a bare repo?"
> Senior dev: "Oh it's just a repo without a working tree, used for server-side hosting."
> Junior dev: *nods while having a stroke*
>
> Translation coming up. It's actually pretty simple.

## 🗄️ Bare Repositories — The Server Copy

A **bare repo** is a Git repository with **no working directory** — it's just the `.git` folder contents.
It's literally what GitHub stores on their servers. You can't edit files in it. You can only push/pull to it.

**Use cases:**
- Self-hosted shared remote (your team's server, NAS, Raspberry Pi — no GitHub needed)
- Git hooks on a server (run scripts when people push)

```bash
# Create a bare repo (on your own server):
git init --bare myproject.git
# Creates: HEAD, config, objects/, refs/ — no actual source files
# The .git extension is convention for bare repos

# Connect your local repo to this bare remote:
git remote add origin user@server:/path/to/myproject.git
git push origin main

# Clone it like any other remote:
git clone user@server:/path/to/myproject.git    # from a server
git clone /path/to/myproject.git  my-working-copy  # locally, same machine
```

> 💬 Regular repo: your project folder has files AND a hidden `.git/` folder inside.
> Bare repo: the WHOLE folder IS the git data. No files to look at, just history.
> GitHub is a massive collection of bare repos with a pretty UI on top.
> Now you know what GitHub actually is. You can be smug about it.

## 📦 Git Submodules — A Repo Inside a Repo

A submodule = a **complete Git repository embedded inside your repository**.

When would you use this?
- Your project depends on a library that's its own Git repo
- You want to track a specific version of an external dependency
- Monorepo situation where multiple components are separate repos but deployed together

Think of it as: `import` for entire Git repositories.

> 💬 Submodules are powerful and also the source of "why is this folder EMPTY??"
> confusion that has haunted many onboarding developers.
> The empty folder means `git submodule update --init` wasn't run.
> Always run it after cloning a repo with submodules.

## 🔧 Submodule Commands

```bash
# Add a submodule to your project:
git submodule add https://github.com/user/library.git libs/library
# Creates: .gitmodules file (tracks what submodules exist) + libs/library/ folder

# After cloning a repo that HAS submodules (the empty folder situation):
git clone --recurse-submodules <url>    # Clone + automatically init all submodules
# OR if you already cloned and subfolders are empty:
git submodule update --init
git submodule update --init --recursive   # For nested submodules (submodules in submodules)

# Update a submodule to its latest commit on its remote:
git submodule update --remote

# See current status of all submodules:
git submodule status

# Remove a submodule properly (4 steps — yes, really):
git submodule deinit libs/library
git rm libs/library
rm -rf .git/modules/libs/library
# Then commit the removal
```

## ⚠️ Submodule Gotchas — Read These Before Using Submodules

- **Submodules are pinned to a SPECIFIC commit**, not a branch. They don't auto-update.
- After cloning any repo, empty submodule folders = `git submodule update --init` not run yet.
- `.gitmodules` tracks which submodules exist — **always commit this file**.
- If you update a submodule, you must ALSO commit the change in the parent repo — else teammates won't see the update.

```bash
# Check your .gitmodules to see what's registered:
cat .gitmodules
# [submodule "libs/library"]
#     path = libs/library
#     url = https://github.com/user/library.git
```

> 💬 Real talk: many experienced developers avoid submodules and use package managers instead.
> npm, pip, cargo, maven — they exist for a reason.
> Submodules are for when a package manager genuinely can't solve the problem.
> Use them intentionally. Not as a default.